# Homework 2: SPY on a 1-minute chart

The assignment has two parts: clean and analyze the data, then build a simple correlation trading rule. I’m using SPY and keeping the main test at 1 minute. There’s enough trading in SPY to try that without starting from 5-minute bars.

I want to see whether an indicator has any relationship with the return that comes **after** it. A smooth line on a chart by itself doesn’t answer that.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 120)
ROOT = Path.cwd()
REBUILD = False
BAR_MINUTES = 1
EMA_MINUTES = 20
RSI_MINUTES = 14
ATR_MINUTES = 14
CORR_MINUTES = 60
CORR_CUTOFF = 0.10
COST_BPS = 0.5

plt.rcParams.update({"figure.dpi": 130, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.18})

## a) Start with the trades

The downloads are huge, so I read one in chunks. Before making bars, I keep SPY trades with a valid time, positive price and size, and no correction/cancel flag. I also remove exact repeated records and prints that are late, out of sequence, or based on an average/reference price. Odd lots stay in: a small trade isn’t automatically a bad trade.

The session here is 9:30 to just before 16:00, New York time. The source timestamps are already Eastern time; they are not UTC. I check the file’s ordering rather than assuming the first row is always the earliest trade.

The chunk-reading code is in `spy_data.py` so the notebook doesn’t turn into a wall of file-handling code. The filter and trading calculations are below.

### Why 1 minute?

Five minutes is a choice, not a requirement. A full session gives 390 one-minute bars versus 78 five-minute bars. I’d rather start at 1 minute and check what happens when I aggregate it later.

The trade-off is noise: a 1-minute move is small, and the spread can eat a lot of it. More bars also doesn’t mean more independent evidence. We still only have a few trading days.

Each bar is labeled by its **start**. The 9:30 bar covers 9:30:00–9:30:59.999…; its close is only known at 9:31. OHLC uses the actual trade order, volume is summed, and VWAP is total price × size divided by total size.

In [ ]:
from spy_data import load_bars

bars, audit = load_bars(ROOT, rebuild=REBUILD)
print(f"{audit['raw_rows']:,} trade records → {audit['kept_rows']:,} kept")
print(f"{len(bars):,} minute bars across {audit['sessions']} sessions")
print(f"{bars.index[0].date()} to {bars.index[-1].date()}")
print("Identical downloads:", audit["duplicate_download"])
print("Times are New York time; no missing minute is filled in.")
display(bars.head().round(4))

audit_table = pd.DataFrame({"records removed": audit["removed"]})
audit_table.loc["kept", "records removed"] = audit["kept_rows"]
display(audit_table.astype(int))

### What’s actually in the sample?

I check the dates instead of calling this “two weeks” from memory. A full-day bar count is useful here: if a minute is missing, I want to see it rather than invent a price for it.

I keep the final five sessions aside for the later test. The earlier sessions are where I inspect the indicators. The window lengths and the 0.10 correlation cutoff above are simple starting choices, not values picked to maximize the final test return.

In [ ]:
sessions = bars.index.normalize().unique().sort_values()
if len(sessions) < 8:
    raise ValueError("Need at least 8 sessions for this split.")
train_days, test_days = sessions[:-5], sessions[-5:]
train_mask = bars.index.normalize().isin(train_days)
test_mask = bars.index.normalize().isin(test_days)

session_summary = bars.groupby(bars.index.normalize()).agg(
    minutes=("close", "size"), trades=("trades", "sum"),
    shares=("volume", "sum"), first_open=("open", "first"), last_close=("close", "last"))
session_summary["open to close (%)"] = 100 * (session_summary["last_close"] / session_summary["first_open"] - 1)
session_summary.index = session_summary.index.strftime("%b %d")
print(f"Earlier sample: {len(train_days)} sessions, {train_days[0].date()}–{train_days[-1].date()}")
print(f"Later test: {len(test_days)} sessions, {test_days[0].date()}–{test_days[-1].date()}")
display(session_summary.round(3))

within_day = bars.groupby(bars.index.normalize())["close"].pct_change(fill_method=None)
training_returns = within_day.loc[train_mask].dropna() * 10_000
print(f"Earlier-sample mean: {training_returns.mean():.3f} bps per minute")
print(f"Earlier-sample std:  {training_returns.std():.3f} bps per minute")
print(f"Lag-1 correlation:  {training_returns.corr(within_day.groupby(bars.index.normalize()).shift(1).loc[train_mask] * 10_000):.3f}")

### How noisy are the minute returns?

These two plots use the earlier sample. Returns are in basis points: 1 bp is 0.01%. The histogram zooms to the middle 99% so a few tails don’t squash everything else; those tails stay in the calculations.

The second plot separates the clock-time pattern from the overnight gaps. The counts in the cleaning table are sequential: each record is counted under the first rule that removes it.

In [ ]:
BLUE, ORANGE, GREEN, RED = "#2463a6", "#d17a22", "#238b68", "#bb4b52"

fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
lo, hi = training_returns.quantile([0.005, 0.995])
axes[0].hist(training_returns.clip(lo, hi), bins=55, color=BLUE, alpha=0.8)
axes[0].axvline(0, color="0.3", lw=1)
axes[0].set(title="1-minute SPY returns · earlier sample", xlabel="Close-to-close return (bps)", ylabel="Minutes")

profile = bars.loc[train_mask].copy()
profile["clock_minute"] = profile.index.hour * 60 + profile.index.minute
profile["abs_return"] = within_day.loc[train_mask].abs() * 10_000
profile = profile.groupby("clock_minute")["abs_return"].mean()
axes[1].plot(profile.index, profile.values, color=ORANGE, lw=1.1)
axes[1].set(xticks=[570, 660, 750, 840, 959], xticklabels=["9:30", "11:00", "12:30", "14:00", "15:59"],
            title="Average move by time of day", xlabel="New York time", ylabel="Mean absolute return (bps)")
plt.show()

### Kalman filter: smooth the signal, keep the prices

I’m using a simple local-level Kalman filter on log price. It takes the previous estimate, adds uncertainty, then adjusts toward the newest close. In the update below, $K$ is the weight on the new observation.

$$P^-_t=P_{t-1}+Q,\qquad K_t=\frac{P^-_t}{P^-_t+R}$$
$$m_t=m_{t-1}+K_t(z_t-m_{t-1}),\qquad P_t=(1-K_t)P^-_t$$

Here $z_t$ is log price, $Q=10^{-8}$, and $R=4\times10^{-8}$. Those are fixed log-price variances, not estimates of the true noise. Their ratio makes the filter less jumpy than the close. I reset it at the start of each day so an overnight move doesn’t get treated as a minute of trading.

I chose Kalman over the usual centered Savitzky–Golay filter because a centered fit uses neighboring points on both sides. A trailing Savitzky–Golay fit would also be possible. Kalman is easier to keep causal here.

This step is smoothing, not deleting “bad” market moves. I don’t interpolate spikes away. The backtest will still use observed trade prices.

In [ ]:
def kalman_price(price, process_var=1e-8, measurement_var=4e-8):
    z = np.log(np.asarray(price, dtype=float))
    estimate = np.empty(len(z))
    estimate[0], variance = z[0], measurement_var
    for i in range(1, len(z)):
        predicted_var = variance + process_var
        weight = predicted_var / (predicted_var + measurement_var)
        estimate[i] = estimate[i - 1] + weight * (z[i] - estimate[i - 1])
        variance = (1 - weight) * predicted_var
    return pd.Series(np.exp(estimate), index=price.index)

bars["kalman"] = bars.groupby(bars.index.normalize())["close"].transform(kalman_price)
example_day = train_days[0]  # Keep the example fixed before looking at the test.
example = bars.loc[bars.index.normalize() == example_day]
zoom = example.between_time("10:00", "11:00")

fig, axes = plt.subplots(2, 1, figsize=(12, 7), constrained_layout=True)
for ax, sample, title in [(axes[0], example, "Full session"), (axes[1], zoom, "10:00–11:00 zoom")]:
    ax.plot(sample.index, sample["close"], color=BLUE, lw=1, alpha=0.6, label="Observed minute close")
    ax.plot(sample.index, sample["kalman"], color=ORANGE, lw=1.8, label="Kalman estimate")
    ax.set(title=f"{example_day.date()} · {title}", ylabel="SPY price ($)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=sample.index.tz))
    ax.legend(loc="upper left", frameon=False, ncol=2)
axes[1].set_xlabel("New York time")
plt.show()

## b) Turn the indicators into something we can correlate

My main input is the distance from the Kalman price to its EMA, divided by ATR. Above the EMA gives a positive value; below gives a negative one. Dividing by ATR makes a $0.20 gap mean less during a volatile stretch.

I also calculate centered RSI and the last price change divided by ATR. RSI is centered at 50. ATR alone has no long/short direction, so the ATR version needs a signed price change.

EMA uses 20 minutes. RSI and ATR use 14 minutes with Wilder’s recursive average, seeded with the first simple average. Flat prices give RSI 50; gains with no losses give 100. All three reset by session and stay unavailable during their warm-up.

In [ ]:
def wilder_mean(values, length):
    result = pd.Series(np.nan, index=values.index, dtype=float)
    valid = values.dropna()
    if len(valid) < length:
        return result
    average = valid.iloc[:length].mean()
    result.loc[valid.index[length - 1]] = average
    for time, value in valid.iloc[length:].items():
        average += (value - average) / length
        result.loc[time] = average
    return result


def make_features(price_bars, minutes=1):
    pieces = []
    ema_n = max(2, int(np.ceil(EMA_MINUTES / minutes)))
    rsi_n = max(2, int(np.ceil(RSI_MINUTES / minutes)))
    atr_n = max(2, int(np.ceil(ATR_MINUTES / minutes)))
    for _, day in price_bars.groupby(price_bars.index.normalize()):
        day = day.copy()
        day["kalman"] = kalman_price(day["close"], process_var=1e-8 * minutes)
        day["ema"] = day["kalman"].ewm(span=ema_n, adjust=False, min_periods=ema_n).mean()
        change = day["kalman"].diff()
        up = wilder_mean(change.clip(lower=0), rsi_n)
        down = wilder_mean(-change.clip(upper=0), rsi_n)
        day["rsi"] = 100 - 100 / (1 + up / down.replace(0, np.nan))
        day.loc[down.eq(0) & up.gt(0), "rsi"] = 100
        day.loc[down.eq(0) & up.eq(0), "rsi"] = 50

        previous_close = day["close"].shift(1)
        true_range = pd.concat([day["high"] - day["low"],
                                (day["high"] - previous_close).abs(),
                                (day["low"] - previous_close).abs()], axis=1).max(axis=1)
        day["atr"] = wilder_mean(true_range, atr_n)
        scale = day["atr"].replace(0, np.nan)
        day["ema_gap"] = (day["kalman"] - day["ema"]) / scale
        day["rsi_centered"] = (day["rsi"] - 50) / 50
        day["atr_move"] = change / scale
        pieces.append(day)
    return pd.concat(pieces)

features = make_features(bars)
FEATURES = {"ema_gap": "EMA distance / ATR", "rsi_centered": "Centered RSI", "atr_move": "Price change / ATR"}
display(features.loc[train_mask, ["ema", "rsi", "atr", *FEATURES]].describe().round(3))

### Correlation with a later return

Correlating price with its EMA would mostly tell me that two price lines move together. I’m interested in the tradable move **after** the indicator instead.

At the end of bar $t$, the feature is $x_t$. The earliest fill I allow is the open of bar $t+1$. I hold until the open of $t+2$, so the target is

$$y_t=\frac{O_{t+2}}{O_{t+1}}-1.$$

By the end of bar $t$, $y_{t-2}$ is known, but $y_t$ isn’t. The rolling Pearson correlation therefore pairs `x.shift(2)` with the open-to-open return that has just finished. It uses the last 60 complete pairs within the same session. Missing warm-up values don’t count.

Positive correlation means the feature and the later return recently lined up. Negative correlation means they tended to oppose each other. A correlation near zero gives me no reason to trade. The sign can change; 0.10 is a trading cutoff, not a claim of statistical significance.

In [ ]:
def add_correlations(feature_frame, minutes=1):
    window = max(3, int(np.ceil(CORR_MINUTES / minutes)))
    pieces = []
    for _, day in feature_frame.groupby(feature_frame.index.normalize()):
        day = day.copy()
        known_return = day["open"].pct_change(fill_method=None)
        for name in FEATURES:
            day[f"corr_{name}"] = day[name].shift(2).rolling(window, min_periods=window).corr(known_return)
        # This forward value is only for the later diagnostic scatterplot.
        day["future_return"] = day["open"].shift(-2) / day["open"].shift(-1) - 1
        pieces.append(day)
    return pd.concat(pieces)

features = add_correlations(features)
earlier_correlations = []
for name, label in FEATURES.items():
    earlier = features.loc[train_mask, [name, "future_return"]].dropna()
    earlier_correlations.append({"input": label, "pairs": len(earlier),
                                 "correlation with later return": earlier[name].corr(earlier["future_return"])})
display(pd.DataFrame(earlier_correlations).set_index("input").round(4))

### Does the relationship show up on a plot?

The dots below are the earlier-sample observations. The orange points are average returns in ten equal-count feature bins. I’m keeping the scatter faint so the averages are readable. A messy, mostly flat cloud is useful information too; it doesn’t need a story forced onto it.

The following chart uses the same fixed example day. Price, RSI, ATR, and correlation get separate axes because their units are different.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.3), constrained_layout=True)
for ax, (name, label) in zip(axes, FEATURES.items()):
    sample = features.loc[train_mask, [name, "future_return"]].dropna().copy()
    sample["return_bps"] = sample["future_return"] * 10_000
    ax.scatter(sample[name], sample["return_bps"], s=8, alpha=0.10, color=BLUE, rasterized=True)
    sample["bin"] = pd.qcut(sample[name], 10, duplicates="drop")
    averages = sample.groupby("bin", observed=True)[[name, "return_bps"]].mean()
    ax.plot(averages[name], averages["return_bps"], "o-", color=ORANGE, lw=2, ms=4, label="Bin average")
    ax.axhline(0, color="0.4", lw=0.8)
    rho = sample[name].corr(sample["future_return"])
    ax.set(title=f"{label} · r = {rho:.3f}", xlabel=label, ylabel="Later open-to-open return (bps)")
    ax.legend(frameon=False, loc="upper right")
fig.suptitle("Indicator now vs. return after the next open · earlier sample", fontsize=14)
plt.show()

example_features = features.loc[features.index.normalize() == example_day]
fig, axes = plt.subplots(4, 1, figsize=(12, 11), sharex=True, constrained_layout=True)
axes[0].plot(example_features.index, example_features["kalman"], color=BLUE, label="Kalman price")
axes[0].plot(example_features.index, example_features["ema"], color=ORANGE, label="EMA · 20 minutes")
axes[0].set(ylabel="Price ($)", title=f"{example_day.date()} · indicators and the main correlation")
axes[0].legend(frameon=False, ncol=2)
axes[1].plot(example_features.index, example_features["rsi"], color=GREEN)
for level in [30, 50, 70]:
    axes[1].axhline(level, color="0.5", ls="--", lw=0.8)
axes[1].set(ylabel="RSI", ylim=(0, 100))
axes[2].plot(example_features.index, example_features["atr"], color=ORANGE)
axes[2].set(ylabel="ATR ($)")
axes[3].plot(example_features.index, example_features["corr_ema_gap"], color=BLUE)
axes[3].axhspan(-CORR_CUTOFF, CORR_CUTOFF, color="0.8", alpha=0.4, label="Stay flat")
axes[3].axhline(0, color="0.4", lw=0.8)
axes[3].set(ylabel="Pearson r", ylim=(-1, 1), xlabel="New York time")
axes[3].legend(frameon=False)
axes[3].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=example_features.index.tz))
plt.show()